In [1]:
import ydf
import pandas as pd
import numpy as np
# run with cleaned dataset and without changing frequency of non IC

In [2]:
crime = pd.read_csv("Crime_Data_from_2020_to_2024.csv")
crime.columns = crime.columns.str.replace(' ', '_')


In [ ]:
# Drop redundant/unnecessary columns
crime= crime.drop(columns=["DR_NO", "Date_Rptd", "AREA","Part_1-2","Crm_Cd", 
                   "Mocodes","Premis_Desc","Weapon_Desc","Status_Desc", 
                   "Crm_Cd_4","LOCATION","Cross_Street","LAT","LON"])
crime = crime.drop(columns=['Crm_Cd_1', 'Crm_Cd_2', 'Crm_Cd_3'])
crime.columns

Index(['DATE_OCC', 'TIME_OCC', 'AREA_NAME', 'Rpt_Dist_No', 'Crm_Cd_Desc',
       'Vict_Age', 'Vict_Sex', 'Vict_Descent', 'Premis_Cd', 'Weapon_Used_Cd',
       'Status'],
      dtype='str')

In [ ]:
# Remove Missing Labels
crime = crime.dropna(subset=['Status'])

In [ ]:
# Set missing values to unknown (X)
crime['Vict_Sex'] = crime['Vict_Sex'].replace(np.nan, "X")
crime['Vict_Sex'] = crime['Vict_Sex'].replace("-", "X")
crime['Vict_Sex'] = crime['Vict_Sex'].replace("H", "X")

crime['Vict_Descent'] = crime['Vict_Descent'].replace(np.nan, "X")
crime['Vict_Descent'] = crime['Vict_Descent'].replace("-", "X")


In [ ]:
# Remove negative ages
crime.loc[crime['Vict_Age'] < 0, 'Vict_Age'] = np.nan


,DATE_OCC,TIME_OCC,AREA_NAME,Rpt_Dist_No,Crm_Cd_Desc,Vict_Age,Vict_Sex,Vict_Descent,Premis_Cd,Weapon_Used_Cd,Status


In [ ]:
def split_dataset(ds, test_ratio):
  test_indices = np.random.rand(len(ds)) < test_ratio
  return ds[~test_indices], ds[test_indices]

crime_train, crime_test = split_dataset(crime, 0.10)
#print(len(crime_train), len(crime_test))


904674 100219


In [19]:
model = ydf.RandomForestLearner(label = "Status", num_trees = 20, missing_value_policy="GLOBAL_IMPUTATION").train(crime_train)

Train model on 904674 examples
Model trained in 0:03:43.182703


In [20]:
model.evaluate(crime_test)

Label \ Pred,IC,AO,AA,JA,JO,CC
IC,77895,6730,6164,260,151,0
AO,1986,3864,1664,35,25,0
AA,291,362,765,19,3,0
JA,0,0,0,4,0,0
JO,0,0,0,0,1,0
CC,0,0,0,0,0,0


In [21]:
model.analyze(crime_test)